In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install faiss-cpu sentence-transformers

import pandas as pd 
import numpy as np 
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

# Load Data
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') # Assuming data is in your local /data/ folder

print("Creating knowledge base...")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index...") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=True) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created!")

# Base Variables for Questions
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 77.0 MB/s eta 0:00:00:00:0100:01
Creating knowledge base...
Loading embedding model and creating index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [2]:
# Q1: Zero-shot classification (No Context)
result_q1 = zs(prompt_150, candidate_labels=labels_150)
idx_correct_q1 = result_q1['labels'].index(ans_150)
prob_q1 = result_q1['scores'][idx_correct_q1]
print(f"Q1 - Probability of correct option (No Context): {prob_q1:.3f}")

# Q2: FAISS Retrieval Rank
prompt_150_embed = model.encode([prompt_150])
distances, retrieved_indices = index.search(prompt_150_embed, 10)
indices_list = list(retrieved_indices[0])

if 150 in indices_list:
    rank_q2 = indices_list.index(150) + 1 # +1 because rank starts at 1
    print(f"Q2 - True document rank via FAISS: {rank_q2}")
else:
    print("Q2 - True document was NOT found in the top 10.")

Q1 - Probability of correct option (No Context): 0.384
Q2 - True document rank via FAISS: 10


Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [3]:
# Q3: Cross-Encoder Reranking
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
docs_10 = [kb[i] for i in indices_list] 
pairs = [[prompt_150, doc] for doc in docs_10] 
ce_scores = cross_encoder.predict(pairs) 

# Sort the retrieved indices based on the Cross-Encoder scores
sorted_indices_by_ce = [x for _, x in sorted(zip(ce_scores, indices_list), reverse=True)]
rank_q3 = sorted_indices_by_ce.index(150) + 1
print(f"Q3 - True document rank after Cross-Encoder Reranking: {rank_q3}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Q3 - True document rank after Cross-Encoder Reranking: 1


Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [4]:
# Q4: Token Count for Concatenated Context
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

prompt_42 = str(train.iloc[42]['prompt'])
prompt_42_embed = model.encode([prompt_42])
_, indices_42 = index.search(prompt_42_embed, 5)

retrieved_docs_42 = [kb[i] for i in indices_42[0]]
concatenated_docs = " ".join(retrieved_docs_42)

rag_string_q4 = f"Context: {concatenated_docs} Question: {prompt_42}"
tokens_q4 = tokenizer(rag_string_q4, truncation=False)['input_ids']

print(f"Q4 - Total number of tokens generated: {len(tokens_q4)}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Q4 - Total number of tokens generated: 216


Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [5]:
# Q5: RAG with the exact TRUE document
true_doc_150 = kb[150]
rag_string_q5 = f"Context: {true_doc_150} Question: {prompt_150}"

result_q5 = zs(rag_string_q5, candidate_labels=labels_150)
idx_correct_q5 = result_q5['labels'].index(ans_150)
prob_q5 = result_q5['scores'][idx_correct_q5]
print(f"Q5 - Probability of correct option (TRUE RAG Context): {prob_q5:.3f}")

# Q6: Adversarial RAG with a WRONG document
wrong_doc = kb[999]
rag_string_q6 = f"Context: {wrong_doc} Question: {prompt_150}"

result_q6 = zs(rag_string_q6, candidate_labels=labels_150)
idx_correct_q6 = result_q6['labels'].index(ans_150)
prob_q6 = result_q6['scores'][idx_correct_q6]
print(f"Q6 - Probability of correct option (ADVERSARIAL RAG Context): {prob_q6:.3f}")

Q5 - Probability of correct option (TRUE RAG Context): 0.989
Q6 - Probability of correct option (ADVERSARIAL RAG Context): 0.529


Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [6]:
# Q7: FAISS Hit Rate for first 100 rows
hits = 0
for i in range(100):
    row = train.iloc[i]
    correct_ans_string = str(row[row['answer']])
    prompt_embed = model.encode([str(row['prompt'])], show_progress_bar=False)
    
    _, retrieved_idx = index.search(prompt_embed, 5)
    retrieved_docs = [kb[idx] for idx in retrieved_idx[0]]
    
    if correct_ans_string in retrieved_docs:
        hits += 1

hit_rate = (hits / 100) * 100
print(f"Q7 - Exact Hit Rate percentage: {hit_rate:.1f}%")

Q7 - Exact Hit Rate percentage: 73.0%


Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [7]:
# Q8: E2E Pipeline MAP@3
def calculate_map3(true_label, top_3_preds):
    if true_label in top_3_preds:
        rank = top_3_preds.index(true_label) + 1
        return 1.0 / rank
    return 0.0

total_map = 0

for i in range(20):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    options = [str(row['A']), str(row['B']), str(row['C']), str(row['D']), str(row['E'])]
    option_letters = ['A', 'B', 'C', 'D', 'E']
    true_ans = row['answer']
    
    # 1. Retrieve
    prompt_embed = model.encode([prompt], show_progress_bar=False)
    _, retrieved_idx = index.search(prompt_embed, 5)
    docs_5 = [kb[idx] for idx in retrieved_idx[0]]
    
    # 2. Rerank
    pairs = [[prompt, doc] for doc in docs_5]
    ce_scores = cross_encoder.predict(pairs)
    best_doc = docs_5[np.argmax(ce_scores)]
    
    # 3. Augment
    rag_string = f"Context: {best_doc} Question: {prompt}"
    
    # 4. Predict
    result = zs(rag_string, candidate_labels=options)
    
    # 5. Score
    # The output 'labels' are the actual strings. We need to map them back to 'A', 'B', 'C', 'D', 'E'
    top_3_strings = result['labels'][:3]
    top_3_letters = [option_letters[options.index(ans_str)] for ans_str in top_3_strings]
    
    total_map += calculate_map3(true_ans, top_3_letters)

average_map3 = total_map / 20
print(f"Q8 - Final Average MAP@3 Score (first 20 rows): {average_map3:.3f}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Q8 - Final Average MAP@3 Score (first 20 rows): 0.975
